In [63]:
import geopandas as gpd
from geopandas import GeoDataFrame
from shapely.geometry import Point, LineString, Polygon
import pandas as pd
from shapely import box

AOI1: str = "../data/fr/img/aoi1.shp"
AOI2: str = "../data/fr/img/aoi2.shp"
OUT: str = "../data/fr/img/extended_overlap.shp"

MIN_X: int = 0
MIN_Y: int = 1
MAX_X: int = 2
MAX_Y: int = 3

In [64]:
aoi1: GeoDataFrame = gpd.read_file(AOI1)
aoi2: GeoDataFrame = gpd.read_file(AOI2)

if aoi1.crs != aoi2.crs:
    aoi2 = aoi2.to_crs(aoi1.crs)

aoi1_bounds = aoi1.total_bounds
aoi2_bounds = aoi2.total_bounds

print(f"AOI1 Data:\n\tBounds: {aoi1_bounds}\n\tCRS: {aoi1.crs}")
print(f"AOI2 Data:\n\tBounds: {aoi2_bounds}\n\tCRS: {aoi2.crs}")

AOI1 Data:
	Bounds: [ 3.62064478 47.58534374  4.29541887 48.16769422]
	CRS: EPSG:4326
AOI2 Data:
	Bounds: [ 4.11711493 47.50028131  4.81887998 48.02783196]
	CRS: EPSG:4326


In [65]:
aoi_intersection = aoi1.intersection(aoi2.union_all())
aoi_intersection_bounds = aoi_intersection.total_bounds

print(f"AOI Intersection Data:\n\tBounds: {aoi_intersection_bounds}\n\tCRS: {aoi_intersection.crs}")

AOI Intersection Data:
	Bounds: [ 4.11711493 47.58534374  4.29337228 48.02783196]
	CRS: EPSG:4326


In [ ]:
width = aoi_intersection_bounds[MAX_X] - aoi_intersection_bounds[MIN_X]
height = aoi_intersection_bounds[MAX_Y] - aoi_intersection_bounds[MIN_Y]

if width >= height:
    print("Horizontal Mosaic")
    overlap_minx = min(aoi1_bounds[MIN_X], aoi2_bounds[MIN_X]) 
    overlap_maxx = max(aoi1_bounds[MAX_X], aoi2_bounds[MAX_X])
    overall_miny = max(aoi1_bounds[MIN_Y], aoi2_bounds[MIN_Y])
    overall_maxy = min(aoi1_bounds[MAX_Y], aoi2_bounds[MAX_Y])

    extended_overlap = box(overlap_minx, overall_miny, overlap_maxx, overall_maxy)
    extended_gdf: GeoDataFrame = GeoDataFrame({'geometry': [extended_overlap]}, crs=aoi1.crs)
    extended_gdf.to_file(OUT)
else:
    print("Vertical Mosaic")
    overlap_minx = max(aoi1_bounds[MIN_X], aoi2_bounds[MIN_X]) 
    overlap_maxx = min(aoi1_bounds[MAX_X], aoi2_bounds[MAX_X])
    overall_miny = min(aoi1_bounds[MIN_Y], aoi2_bounds[MIN_Y])
    overall_maxy = max(aoi1_bounds[MAX_Y], aoi2_bounds[MAX_Y])

    extended_overlap = box(overlap_minx, overall_miny, overlap_maxx, overall_maxy)
    extended_gdf: GeoDataFrame = GeoDataFrame({'geometry': [extended_overlap]}, crs=aoi1.crs)
    extended_gdf.to_file(OUT)


Vertical Mosaic
